# Тест агента — 4 игрока (FFA)

**наш агент** (P0) против трёх оппонентов в режиме Free-For-All.

| Имя | Описание |
|-----|----------|
| `sub2` | Стандартный Kaggle-бенчмарк. Имеет `is_four_player` логику |
| `opp_exposed` | Топ-решение. FFA-aware: TIE_HUNT, TERMINAL_LAUNCH |
| `opp_1101` | exposed без эндшпиля |
| `opp_heuristic` | Простой эвристик. Для smoke-тестов |
| `noop` | Пустой агент |

Подробнее → `OPPONENTS.md`

> **Механика FFA**: победитель — игрок с наибольшим reward в конце.
> При равных — все получают одинаковое вознаграждение (ничья).
> `sub2` и `opp_exposed` имеют специальную FFA-логику (LET_THEM_FIGHT, CRASH_EXPLOIT отключён).

In [ ]:
import importlib.util, sys, os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from kaggle_environments import make

HERE     = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
OUR_PATH = os.path.join(HERE, "agent_bundle_swarm 2", "agent.py")

ALL_OPPONENTS = {
    'sub2':          os.path.join(HERE, 'sub2.py'),
    'opp_exposed':   os.path.join(HERE, 'opp_exposed.py'),
    'opp_1101':      os.path.join(HERE, 'opp_1101.py'),
    'opp_heuristic': os.path.join(HERE, 'opp_heuristic.py'),
}

def load_agent(path, name):
    if name in sys.modules: del sys.modules[name]
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod.agent

our_agent = load_agent(OUR_PATH, 'our_agent')

# ═══════════════════════════════════════════════════════════════
# ВЫБЕРИ ТРЁХ ОППОНЕНТОВ — меняй здесь
OPPS = ['sub2', 'sub2', 'sub2']   # [P1, P2, P3]
# ═══════════════════════════════════════════════════════════════

SEED = 42

print("our_agent:", our_agent)
print("OPPS     :", OPPS)
print("SEED     :", SEED)
print("\nДоступные оппоненты:")
for name, path in ALL_OPPONENTS.items():
    exists = '✓' if os.path.exists(path) else '✗ НЕ НАЙДЕН'
    print(f'  {name:<18} {exists}')


## Один матч — дебаг и визуализация

Запускает один матч с заданным `SEED` и строит графики по всем четырём игрокам.

In [ ]:
OUR_LOG = os.path.join(HERE, 'agent_debug.log')
if os.path.exists(OUR_LOG): os.remove(OUR_LOG)
os.environ['ORBIT_AGENT_LOG']           = OUR_LOG
os.environ['ORBIT_AGENT_LOG_PLANS_ALL'] = '1'
os.environ['ORBIT_AGENT_LOG_FLEETS']    = '0'

for _m in ('our_agent', 'agent_debug') + tuple(f'_opp{i}' for i in range(3)):
    if _m in sys.modules: del sys.modules[_m]

our_agent_dbg = load_agent(OUR_PATH, 'our_agent')

def _load_opp_direct(name, mod_name):
    if name == 'noop': return lambda obs, cfg=None: []
    return load_agent(ALL_OPPONENTS[name], mod_name)

opp_agents = [_load_opp_direct(name, f'_opp{i}') for i, name in enumerate(OPPS)]

env_dbg = make('orbit_wars', debug=False, configuration={'seed': SEED})
env_dbg.run([our_agent_dbg] + opp_agents)

final_dbg = env_dbg.steps[-1]
rewards   = [float(s.get('reward') or 0) for s in final_dbg]
rank      = 1 + sum(1 for r in rewards[1:] if r > rewards[0])
labels    = ['our (P0)'] + [f'P{i+1}: {OPPS[i]}' for i in range(3)]

print(f'Ходов: {len(env_dbg.steps)}   Наш ранг: {rank}/4')
for i, (lbl, rew) in enumerate(zip(labels, rewards)):
    marker = ' ← WE' if i == 0 else ''
    print(f'  P{i} {lbl:<22} reward={rew:.3f}{marker}')

log_size = os.path.getsize(OUR_LOG) if os.path.exists(OUR_LOG) else 0
print(f'\nЛог нашего агента: {OUR_LOG}  ({log_size:,} байт)')

env_dbg.render(mode='ipython', width=900, height=650)


## Графики одного матча

In [ ]:
def extract_ts4(env_obj):
    """Извлекает timeseries для 4 игроков из env."""
    ts_ships = [[] for _ in range(4)]
    ts_prod  = [[] for _ in range(4)]
    for step in env_obj.steps:
        obs     = step[0].get('observation') or {}
        planets = obs.get('planets') or []
        fleets  = obs.get('fleets')  or []
        ships = [0.0] * 4
        prod  = [0.0] * 4
        for p in planets:
            own  = p[1] if isinstance(p, list) else p.get('owner', -1)
            shps = float(p[5] if isinstance(p, list) else p.get('ships', 0) or 0)
            prdn = float(p[6] if isinstance(p, list) else p.get('production', 0) or 0)
            if 0 <= own < 4: ships[own] += shps; prod[own] += prdn
        for f in fleets:
            own  = f[1] if isinstance(f, list) else f.get('owner', -1)
            shps = float(f[6] if isinstance(f, list) else f.get('ships', 0) or 0)
            if 0 <= own < 4: ships[own] += shps
        for i in range(4):
            ts_ships[i].append(ships[i])
            ts_prod[i].append(prod[i])
    return ts_ships, ts_prod

COLORS4 = ['tab:blue', 'tab:red', 'tab:green', 'tab:orange']

ts_ships_dbg, ts_prod_dbg = extract_ts4(env_dbg)
labels = ['our (P0)'] + [f'P{i+1}: {OPPS[i]}' for i in range(3)]
steps  = list(range(len(ts_ships_dbg[0])))

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Корабли
for i in range(4):
    lw = 2.5 if i == 0 else 1.2
    axes[0].plot(steps, ts_ships_dbg[i], label=labels[i], color=COLORS4[i], linewidth=lw)
axes[0].set_title('Суммарные корабли (включая флоты)')
axes[0].set_xlabel('ход'); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Производство
for i in range(4):
    lw = 2.5 if i == 0 else 1.2
    axes[1].plot(steps, ts_prod_dbg[i], label=labels[i], color=COLORS4[i], linewidth=lw)
axes[1].set_title('Производство')
axes[1].set_xlabel('ход'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# Ships ratio нашего агента ко всем
total_ships = [sum(ts_ships_dbg[j][t] for j in range(4)) for t in steps]
our_ratio   = [ts_ships_dbg[0][t] / max(1, total_ships[t]) for t in steps]
axes[2].plot(steps, our_ratio, color='tab:blue', linewidth=2)
axes[2].axhline(0.25, color='gray', linestyle='--', linewidth=0.8, label='равный 1/4')
axes[2].fill_between(steps, 0.25, our_ratio,
                     where=[r > 0.25 for r in our_ratio], alpha=0.15, color='green')
axes[2].fill_between(steps, 0.25, our_ratio,
                     where=[r < 0.25 for r in our_ratio], alpha=0.15, color='red')
axes[2].set_title('Наша доля кораблей (> 0.25 = лидируем)')
axes[2].set_xlabel('ход'); axes[2].set_ylim(0, 1); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f'Seed {SEED}  |  Ранг нашего агента: {rank}/4', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## Параллельный батч

Запускает N матчей параллельно через `match_runner4.run_match4`.
Результаты: ранг, top-2, winrate, timeseries по всем 4 игрокам.

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm.notebook import tqdm
import time

if HERE not in sys.path: sys.path.insert(0, HERE)
from match_runner4 import run_match4

# ── КОНФИГ БАТЧА ─────────────────────────────────────────────────────────
BATCH_SEEDS = list(range(30))       # ← сиды
BATCH_OPPS  = OPPS                  # ← берём из первой ячейки; или задать вручную
N_WORKERS   = max(2, os.cpu_count() // 2)
WEIGHTS     = {}                    # {} = дефолты SwarmWeights
# ─────────────────────────────────────────────────────────────────────────

tasks = [
    {'seed': s, 'our_path': OUR_PATH, 'opps': BATCH_OPPS,
     'weights': WEIGHTS, 'label': '+'.join(BATCH_OPPS)}
    for s in BATCH_SEEDS
]

print(f'Задач: {len(tasks)}  |  воркеров: {N_WORKERS}  |  оппоненты: {BATCH_OPPS}')


In [ ]:
# ── ЗАПУСК ──────────────────────────────────────────────────────────────
results4 = []
errors4  = []
t0 = time.time()

with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futs = {ex.submit(run_match4, t): t for t in tasks}
    for fut in tqdm(as_completed(futs), total=len(tasks), desc='матчи 4p'):
        try:
            results4.append(fut.result())
        except Exception as e:
            task = futs[fut]
            errors4.append((task['seed'], str(e)))
            print(f'  ERROR seed={task["seed"]}: {e}')

elapsed = time.time() - t0
df4 = pd.DataFrame(results4).sort_values('seed').reset_index(drop=True)

n      = len(df4)
wins   = int(df4['win'].sum())
top2   = int(df4['top2'].sum())
avg_rank = df4['rank'].mean()
print(f'Готово за {elapsed:.1f}s  ({elapsed/max(n,1):.1f}s/матч)')
print(f'Winrate (1е место): {wins}/{n} = {wins/max(n,1):.1%}')
print(f'Top-2 rate:         {top2}/{n} = {top2/max(n,1):.1%}')
print(f'Avg rank:           {avg_rank:.2f} / 4  (1 = лучший)')
if errors4:
    print(f'Ошибок: {len(errors4)}')
    for seed, msg in errors4: print(f'  seed={seed}: {msg}')


## Результаты батча

In [ ]:
# ── СВОДНАЯ ТАБЛИЦА ──────────────────────────────────────────────────────
summary4 = pd.DataFrame({
    'метрика': [
        'матчей', 'побед (1е место)', 'top-2', 'поражений (4е место)',
        'avg rank', 'winrate', 'top-2 rate',
        'avg steps', 'avg ships P0 final', 'avg prod P0 final',
    ],
    'значение': [
        n, wins, top2, int((df4['rank'] == 4).sum()),
        f'{avg_rank:.2f}', f'{wins/max(n,1):.1%}', f'{top2/max(n,1):.1%}',
        f'{df4["steps"].mean():.1f}',
        f'{df4["ships_final_0"].mean():.0f}',
        f'{df4["prod_final_0"].mean():.1f}',
    ]
})
display(summary4.set_index('метрика'))

# Детальная таблица по сидам
cols = ['seed', 'rank', 'win', 'top2', 'steps',
        'reward_0', 'reward_1', 'reward_2', 'reward_3',
        'planets_final_0', 'ships_final_0', 'prod_final_0', 'time_sec']
def rank_color(val):
    if val == 1: return 'background-color: #d4edda'
    if val == 2: return 'background-color: #fff3cd'
    if val == 4: return 'background-color: #f8d7da'
    return ''
display(df4[cols].style.applymap(rank_color, subset=['rank']))


In [ ]:
# ── РАСПРЕДЕЛЕНИЕ РАНГОВ ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Гистограмма рангов
rank_counts = df4['rank'].value_counts().sort_index()
colors_rank = ['#2ecc71', '#f1c40f', '#e67e22', '#e74c3c']
axes[0].bar(rank_counts.index, rank_counts.values,
            color=[colors_rank[i-1] for i in rank_counts.index])
axes[0].set_xticks([1,2,3,4])
axes[0].set_xticklabels(['1е\n(победа)', '2е', '3е', '4е\n(поражение)'])
axes[0].set_title(f'Распределение мест  (n={n})')
axes[0].set_ylabel('матчей')
for bar, cnt in zip(axes[0].patches, rank_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.2,
                 f'{cnt}', ha='center', fontsize=10)
axes[0].grid(axis='y', alpha=0.3)

# Avg ships ratio по ходам (усреднённые по всем матчам)
max_steps = max(len(r['ts_ships_0']) for _, r in df4.iterrows())
COLORS4 = ['tab:blue','tab:red','tab:green','tab:orange']
labels4  = ['our (P0)'] + [f'P{i+1}: {BATCH_OPPS[i]}' for i in range(3)]
for pid in range(4):
    avg_ts = []
    for t in range(max_steps):
        vals = [r[f'ts_ships_{pid}'][t] for _, r in df4.iterrows()
                if t < len(r[f'ts_ships_{pid}'])]
        avg_ts.append(sum(vals)/len(vals) if vals else 0)
    lw = 2.5 if pid == 0 else 1.2
    axes[1].plot(avg_ts, label=labels4[pid], color=COLORS4[pid], linewidth=lw)
axes[1].set_title('Avg корабли по ходам')
axes[1].set_xlabel('ход'); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

# Avg наша доля кораблей
avg_ratio = []
for t in range(max_steps):
    ratios = []
    for _, r in df4.iterrows():
        if t < len(r['ts_ships_0']):
            tot = sum(r[f'ts_ships_{j}'][t] for j in range(4))
            ratios.append(r['ts_ships_0'][t] / max(1, tot))
    avg_ratio.append(sum(ratios)/len(ratios) if ratios else 0)
axes[2].plot(avg_ratio, color='tab:blue', linewidth=2)
axes[2].axhline(0.25, color='gray', linestyle='--', linewidth=0.8, label='1/4 (равный)')
axes[2].fill_between(range(len(avg_ratio)), 0.25, avg_ratio,
                     where=[r > 0.25 for r in avg_ratio], alpha=0.15, color='green')
axes[2].fill_between(range(len(avg_ratio)), 0.25, avg_ratio,
                     where=[r < 0.25 for r in avg_ratio], alpha=0.15, color='red')
axes[2].set_title('Avg доля наших кораблей')
axes[2].set_xlabel('ход'); axes[2].set_ylim(0, 0.6)
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle(f'Батч {n} матчей  |  оппоненты: {BATCH_OPPS}', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()


## Детальные графики по матчам

Разворачивает timeseries каждого матча. Удобно для анализа провальных сидов.

In [ ]:
# ── PER-SEED TIMESERIES ──────────────────────────────────────────────────
n_show = min(len(df4), 15)   # ← максимум матчей для отображения
df4_show = df4.head(n_show)

fig, axes = plt.subplots(n_show, 2, figsize=(15, 3.5 * n_show), squeeze=False)
COLORS4  = ['tab:blue','tab:red','tab:green','tab:orange']
labels4  = ['our (P0)'] + [f'P{i+1}: {BATCH_OPPS[i]}' for i in range(3)]

for row_i, (_, r) in enumerate(df4_show.iterrows()):
    rank_str  = f'#{r["rank"]}/4'
    win_str   = {1:'WIN', 2:'TOP2', 3:'3rd', 4:'LOSS'}[r['rank']]
    tc        = {'WIN':'tab:green','TOP2':'tab:olive','3rd':'tab:orange','LOSS':'tab:red'}[win_str]
    ax_s = axes[row_i, 0]
    ax_p = axes[row_i, 1]
    for pid in range(4):
        lw = 2.5 if pid == 0 else 1.0
        ax_s.plot(r[f'ts_ships_{pid}'], label=labels4[pid], color=COLORS4[pid], linewidth=lw)
        ax_p.plot(r[f'ts_prod_{pid}'],  label=labels4[pid], color=COLORS4[pid], linewidth=lw)
    ax_s.set_title(f'Seed {r["seed"]} — {win_str} {rank_str} — корабли', color=tc)
    ax_s.set_xlabel('ход'); ax_s.legend(fontsize=7); ax_s.grid(alpha=0.3)
    ax_p.set_title(f'Seed {r["seed"]} — производство')
    ax_p.set_xlabel('ход'); ax_p.legend(fontsize=7); ax_p.grid(alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# ── СОХРАНИТЬ РЕЗУЛЬТАТЫ ────────────────────────────────────────────────
from datetime import datetime
ts_str  = datetime.now().strftime('%Y%m%d_%H%M%S')
out_dir = os.path.join(HERE, 'out')
os.makedirs(out_dir, exist_ok=True)
save_cols = [c for c in df4.columns if not c.startswith('ts_')]
opp_tag   = '+'.join(BATCH_OPPS).replace('opp_', '')
out_csv   = os.path.join(out_dir, f'batch4_{opp_tag}_{ts_str}.csv')
df4[save_cols].to_csv(out_csv, index=False)
print(f'Сохранено: {out_csv}')
